# 01 · Grilla de 5 km y municipios

**Qué hace.** Arma la grilla de celdas de 5 × 5 km que cubre Colombia, que es la unidad de todas las tablas
de deforestación, y le asigna a cada celda su departamento y su municipio con los límites oficiales del DANE.
Los límites se usan solo de paso y no se guardan en BigQuery: los mapas de los notebooks 09 y 12 los vuelven
a descargar del DANE.

**Cuándo correrlo.** Una vez, antes que todos los demás. Solo hay que repetirlo si el DANE cambia los
límites o si se quiere otra grilla.

**Qué lee.** Los límites de departamentos y municipios del DANE (Marco Geoestadístico Nacional), de su
servicio público en internet. No necesita ninguna tabla.

**Qué escribe**, en el dataset `DATASET_DEFORESTACION`:

| Tabla | Qué es |
|---|---|
| `GRILLA` | Una fila por celda: identificador, centro, departamento y municipio |

Si una versión anterior de este notebook dejó la tabla `MUNICIPIOS_LIMITES` en el dataset, la borra.

**Qué tienes que cambiar.** Nada: la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`).

**Cuánto tarda.** Unos dos minutos.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Parámetros de la grilla ------------------------------------------
# Son los del trabajo de grado y no se cambian: todas las tablas de
# deforestación dependen de ellos.
LADO_CELDA_M = 5_000          # lado de cada celda, en metros (5 km)
CRS_GRILLA = "EPSG:3116"      # MAGNA-SIRGAS / Colombia Bogotá, en metros

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet geopandas==1.1.1 shapely pyproj requests pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "01_carga_grilla_y_municipios"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

## 4. Límites de los departamentos

Se piden al servicio público del DANE (Marco Geoestadístico Nacional 2023, nivel departamento) ya en la
proyección de la grilla, EPSG:3116: el servidor reproyecta antes de enviar. Licencia CC BY 4.0: cualquier
producto que use estas tablas debe citar "Departamento Administrativo Nacional de Estadística - DANE".

In [ ]:
import geopandas as gpd
import requests
import shapely.geometry as sg

# El DANE renombró este servicio en 2026; antes era
# "MarcoGeoestadisticoNacional2023_NivelDepartamento". Los datos son los mismos.
URL_DEPARTAMENTOS = ("https://services.arcgis.com/ioNRSZMYYlx0PUaB/arcgis/rest/services/"
                     "MGN2023_DPTO_POLITICO/FeatureServer/0/query")

r = requests.get(URL_DEPARTAMENTOS, params={
    "where": "1=1",
    "outFields": "dpto_ccdgo,dpto_cnmbr",
    "outSR": CRS_GRILLA.split(":")[-1],      # "EPSG:3116" -> "3116"
    "f": "geojson",
}, timeout=300)
r.raise_for_status()

departamentos = gpd.GeoDataFrame.from_features(r.json()["features"], crs=CRS_GRILLA)
departamentos = departamentos.rename(columns={"dpto_cnmbr": "departamento", "dpto_ccdgo": "cod_dpto"})
# El DANE escribe los nombres en mayúsculas ("ANTIOQUIA"); se dejan legibles.
departamentos["departamento"] = departamentos["departamento"].str.title()
print(f"{len(departamentos)} departamentos descargados")

## 5. La grilla

Cada celda es el cuadrado `[ix·5000, (ix+1)·5000) × [iy·5000, (iy+1)·5000)` en metros, para un par de enteros
`(ix, iy)`. Así la grilla queda alineada con el origen de la proyección por construcción, y saber en qué
celda cae un punto es una división: `floor(x / 5000)`. Los notebooks de Hansen, IDEAM y detecciones
tempranas dependen de esa regla.

Se arman todas las celdas del rectángulo que envuelve a Colombia y se dejan las que tocan el país. Para ese
filtro se usa el contorno del país simplificado a 100 m, porque el del DANE tiene cientos de miles de
vértices y el filtro se vuelve muy lento. Mover un borde 100 m no saca ni mete ninguna celda de 5 km.

In [ ]:
import numpy as np
import pandas as pd

pais = departamentos.geometry.union_all().simplify(100, preserve_topology=True)
lado = LADO_CELDA_M
minx, miny, maxx, maxy = pais.bounds
ix0, ix1 = int(np.floor(minx / lado)), int(np.ceil(maxx / lado))
iy0, iy1 = int(np.floor(miny / lado)), int(np.ceil(maxy / lado))
print(f"Celdas candidatas: {ix1 - ix0} x {iy1 - iy0} = {(ix1 - ix0) * (iy1 - iy0):,}")

IX, IY = np.meshgrid(np.arange(ix0, ix1), np.arange(iy0, iy1))
IX, IY = IX.ravel(), IY.ravel()
cajas = [sg.box(i * lado, j * lado, (i + 1) * lado, (j + 1) * lado) for i, j in zip(IX, IY)]
candidatas = gpd.GeoDataFrame({"ix": IX, "iy": IY}, geometry=cajas, crs=CRS_GRILLA)

# sjoin usa un índice espacial, por eso es viable con cientos de miles de candidatas.
celdas = gpd.sjoin(candidatas, gpd.GeoDataFrame(geometry=[pais], crs=CRS_GRILLA),
                   predicate="intersects", how="inner")
celdas = celdas.drop(columns=["index_right"]).drop_duplicates(["ix", "iy"]).reset_index(drop=True)
print(f"Celdas que tocan Colombia: {len(celdas):,}")

### 5.1 Centro, identificador y departamento de cada celda

- El centro de cada celda está a medio lado de sus bordes, así que sale de `ix` e `iy` sin cálculo
  geométrico.
- El identificador `cell_id` es la longitud y la latitud del centro multiplicadas por 10.000 y redondeadas.
  Es el mismo criterio de todas las versiones del proyecto, así que los identificadores son estables.
- El departamento es el polígono del DANE que contiene el **centro** de la celda: una celda partida entre
  dos departamentos va al de su centro.

In [ ]:
x3116 = (celdas["ix"] + 0.5) * lado
y3116 = (celdas["iy"] + 0.5) * lado
centros = gpd.GeoSeries(gpd.points_from_xy(x3116, y3116), crs=CRS_GRILLA)
centros_4326 = centros.to_crs("EPSG:4326")
lon, lat = centros_4326.x, centros_4326.y

cell_id = ((lon * 1e4).round().astype("int64").astype(str) + "_"
           + (lat * 1e4).round().astype("int64").astype(str))

grilla = pd.DataFrame({
    "cell_id": cell_id,
    "lon": lon.values, "lat": lat.values,
    "x3116": x3116.values, "y3116": y3116.values,
    "ix": celdas["ix"].values, "iy": celdas["iy"].values,
})
puntos = gpd.GeoDataFrame(grilla, geometry=centros.values, crs=CRS_GRILLA)
unido = gpd.sjoin(puntos, departamentos[["departamento", "geometry"]], predicate="within", how="left")
grilla["departamento"] = unido["departamento"].values
grilla["ix"] = grilla["ix"].astype("int32")
grilla["iy"] = grilla["iy"].astype("int32")

### 5.2 Controles

1. **Alineación.** Si la grilla está alineada con el origen, el centro de cada celda cae justo a 2.500 m de
   sus bordes. Si no, el notebook se detiene: los demás asignarían los píxeles a la celda equivocada.
2. **Identificadores repetidos.** No debería haber; si los hay, se deja el primero.
3. **Celdas sin departamento.** Son celdas de costa cuyo centro cae en el mar. Se descartan: no tienen
   bosque que medir.

In [ ]:
rx = (grilla["x3116"] % lado).round(1)
ry = (grilla["y3116"] % lado).round(1)
alineada = (rx - lado / 2).abs().max() < 1.0 and (ry - lado / 2).abs().max() < 1.0
print(f"Resto de x: {rx.min():.1f} a {rx.max():.1f}; resto de y: {ry.min():.1f} a {ry.max():.1f} "
      f"(esperado {lado / 2:.1f})")
if not alineada:
    raise RuntimeError("La grilla no quedó alineada con el origen de EPSG:3116. No sigas: "
                       "las tablas de deforestación saldrían mal.")
print("Grilla alineada con el origen: correcto.")

repetidos = grilla["cell_id"].duplicated().sum()
if repetidos:
    print(f"Aviso: {repetidos} cell_id repetidos; se deja el primero.")
    grilla = grilla.drop_duplicates("cell_id")

sin_departamento = grilla["departamento"].isna().sum()
if sin_departamento:
    print(f"{sin_departamento} celdas sin departamento (centro en el mar); se descartan.")
    grilla = grilla.dropna(subset=["departamento"])
grilla = grilla.reset_index(drop=True)
print(f"Grilla final: {len(grilla):,} celdas en {grilla['departamento'].nunique()} departamentos")

## 6. Municipio de cada celda

Con los límites municipales del DANE (Marco Geoestadístico Nacional 2025, versión "Gráfico", que es la
generalizada para mapas) se asigna a cada celda el municipio que contiene su centro. Esa versión basta para
esto: su simplificación es miles de veces menor que una celda de 5 km. El código DANE (`cod_dane`) es la
llave para unir la deforestación con las tablas del Observatorio; el nombre es solo para leer.

Los paneles de deforestación toman de aquí el municipio de cada celda, así que este cruce se hace una sola
vez.

In [ ]:
URL_MUNICIPIOS = ("https://services.arcgis.com/ioNRSZMYYlx0PUaB/arcgis/rest/services/"
                  "MGN2025_MPIO_GRAFICO/FeatureServer/0/query")

r = requests.get(URL_MUNICIPIOS, params={
    "where": "1=1",
    "outFields": "mpio_cdpmp,mpio_cnmbr,dpto_cnmbr,dpto_ccdgo",
    "outSR": "4326",
    "f": "geojson",
}, timeout=300)
r.raise_for_status()
municipios_dane = r.json()
if not municipios_dane.get("features"):
    raise RuntimeError("El servicio del DANE respondió sin municipios: " + str(municipios_dane)[:300])
mgn = gpd.GeoDataFrame.from_features(municipios_dane["features"], crs="EPSG:4326")
print(f"{len(mgn)} municipios descargados")

centros_celda = gpd.GeoDataFrame(grilla[["cell_id"]],
                                 geometry=gpd.points_from_xy(grilla.lon, grilla.lat), crs="EPSG:4326")
cruce = gpd.sjoin(centros_celda, mgn[["mpio_cdpmp", "mpio_cnmbr", "geometry"]], how="left", predicate="within")
if cruce["cell_id"].duplicated().any():
    # Pasaría si un centro cayera justo sobre el borde de dos municipios.
    print(f"Aviso: {cruce['cell_id'].duplicated().sum()} celdas en dos municipios; se deja el primero.")
    cruce = cruce.drop_duplicates("cell_id")
cruce = cruce.set_index("cell_id")
grilla["cod_dane"] = grilla["cell_id"].map(cruce["mpio_cdpmp"])
# El DANE escribe los nombres en mayúsculas ("MEDELLÍN"); se dejan legibles.
grilla["municipio"] = grilla["cell_id"].map(cruce["mpio_cnmbr"]).str.title()

print(f"Celdas con municipio: {grilla['cod_dane'].notna().sum():,} de {len(grilla):,} "
      f"en {grilla['cod_dane'].nunique():,} municipios")
grilla.head()

## 7. Guardar en BigQuery

La grilla se guarda con una descripción y con la explicación de cada columna, que se ven en BigQuery en la
pestaña de esquema. Los polígonos de los municipios no se guardan: el Observatorio ya los tiene.

In [ ]:
guardar_tabla(
    grilla, "grilla",
    "Grilla de celdas de 5 x 5 km sobre Colombia, alineada con el origen de EPSG:3116. "
    "Es la unidad de todas las tablas de deforestación. Fuente de los límites: DANE, "
    "Marco Geoestadístico Nacional 2023 (departamentos) y 2025 (municipios), CC BY 4.0.",
    descripciones={
        "cell_id": "Identificador de la celda: longitud y latitud del centro por 10.000, redondeadas. "
                   "Llave para unir las tablas por celda.",
        "lon": "Longitud del centro de la celda (grados, EPSG:4326).",
        "lat": "Latitud del centro de la celda (grados, EPSG:4326).",
        "x3116": "Coordenada X del centro, en metros (EPSG:3116).",
        "y3116": "Coordenada Y del centro, en metros (EPSG:3116).",
        "ix": "Columna de la celda en la grilla: floor(x / 5000).",
        "iy": "Fila de la celda en la grilla: floor(y / 5000).",
        "departamento": "Departamento donde cae el centro de la celda (DANE, MGN 2023).",
        "cod_dane": "Código DANE de 5 dígitos del municipio donde cae el centro (DANE, MGN 2025). "
                    "Vacío si el centro cae fuera de todo municipio.",
        "municipio": "Nombre del municipio, solo para leer. Para cruzar con otras tablas usa cod_dane.",
    },
    procedencia={"fuente": "DANE MGN 2023 (departamentos) y MGN 2025 Gráfico (municipios)",
                 "lado_celda_m": LADO_CELDA_M, "crs": CRS_GRILLA})

# Una versión anterior de este notebook guardaba los límites de los municipios
# en la tabla MUNICIPIOS_LIMITES. Ya no se guardan; si quedó, se borra.
for _vieja in ("MUNICIPIOS_LIMITES", "municipios_limites"):
    _tabla = f"{PROYECTO}.{DATASET_DEFORESTACION}.{_vieja}"
    try:
        client.get_table(_tabla)
    except Exception:
        continue
    client.delete_table(_tabla)
    print(f"Borrada {_tabla}: los límites municipales ya no se guardan en BigQuery.")

Listo. Sigue `02_carga_hansen`.